# Fase 15 — Matriz / Score de Confiança (`notebooks/15_confidence_matrix.ipynb`)

**Pergunta:** além de prever um preço, o sistema consegue dizer "quão confiável é ESTA previsão em
particular" — de forma calibrada, não um chute?

Esta é uma extensão pós-plano sobre a API (`docs/02_execution_plan_api.md`): não estava no plano
original de implementação — foi pedida depois, como uma camada de confiança por predição, sobre os
splits `TEST`/`VAL` já existentes. Segue a mesma disciplina metodológica do resto do projeto
(P1/P4/P5): **`TEST` calibra, `VAL` verifica uma única vez, nada é reajustado depois de ver o
resultado.** Ver `docs/09_confidence_matrix.md` para a explicação end-to-end completa (este notebook
narra a mesma execução, com os números desta rodada).

**Metodologia (resumo, ver `src/evaluation/confidence.py` e `docs/09_confidence_matrix.md` para os
detalhes):**

- **`TEST` (via `model_candidate.pkl`, treinado só em `train`, nunca viu `test`):** mede erro real por
  segmento (`price_band` × `property_cluster` × cobertura no espaço de features ×
  `waterfront`/`grade≥10`, com backoff em cascata para amostras pequenas, `n < 20`) — nunca um MAE
  único (P3). Constrói o **Confidence Score (0-100)** como rank empírico (ECDF) do erro esperado do
  segmento vs. toda a distribuição de `test`. Calibra a curva score → erro esperado via **regressão
  isotônica** (monótona, sem peso escolhido à mão) e deriva os **cortes de categoria** dos quartis
  REAIS do erro já calibrado em `test` (P25/P50/P75) — não de um número "que parece bom". Tudo isso
  fica congelado em `artifacts/confidence_calibration.json`.
- **`VAL` (via `model_final.pkl`, treinado em `train`+`test`, tocado nesta checagem):** aplica a
  calibração congelada sem recalcular nada — a confiança prometida em `test` realmente se sustenta em
  dado nunca usado para calibrar?
- Além das 4 categorias finas, uma **matriz 2×2 didática** (cobertura alta/baixa × erro esperado
  baixo/alto, cortes na mediana de `test`) — versão mais simples e robusta da mesma ideia.
- Uma **checagem de monotonicidade** explícita: o erro observado em `VAL` realmente cresce (ou empata)
  de "Alta confiança" para "Muito baixa confiança / revisão", na mesma ordem prometida pela calibração
  em `test`?

Este script (`src/scripts/build_confidence_matrix.py`) era **órfão** — existia desde antes, mas nenhum
notebook o chamava; é adotado aqui. **Não é uma fase de decisão de modelo/feature (P4)** — não muda
`model_final.pkl` nem o conjunto de 20 features.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
from src.scripts.build_confidence_matrix import run_confidence_matrix_pipeline

report = run_confidence_matrix_pipeline()
print("n_test:", report["n_test"], "| n_val:", report["n_val"])
print("category_thresholds:", report["category_thresholds"])
print("ape_breakpoints_from_test:", report["ape_breakpoints_from_test"])
print("is_monotonic_by_category (VAL):", report["verification_on_val"]["is_monotonic_by_category"])

In [ ]:
import pandas as pd

print("Matriz de confiança 2x2 — calibrada em TEST:")
matrix_test_df = pd.DataFrame(report["calibration_on_test"]["matrix_2x2"])
display(matrix_test_df)

print("\nMatriz de confiança 2x2 — verificada em VAL (cortes de distance/expected_ape CONGELADOS de TEST):")
matrix_val_df = pd.DataFrame(report["verification_on_val"]["matrix_2x2"])
display(matrix_val_df)

print("\n4 categorias finas — calibração em TEST:")
by_cat_test_df = pd.DataFrame(report["calibration_on_test"]["by_category"])
display(by_cat_test_df)

print("\n4 categorias finas — verificação em VAL (com teto prometido em TEST e se foi respeitado):")
by_cat_val_df = pd.DataFrame(report["verification_on_val"]["by_category"])
display(by_cat_val_df)

In [ ]:
import json

calibration_artifact = json.loads(Path("artifacts/confidence_calibration.json").read_text(encoding="utf-8"))
curve_df = pd.DataFrame(calibration_artifact["isotonic_curve"], columns=["confidence_score", "calibrated_ape"])
print("Curva de calibração isotônica (score 0-100 -> APE calibrado), primeiras/últimas linhas:")
display(pd.concat([curve_df.head(5), curve_df.tail(5)]))

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

thresholds = report["category_thresholds"]

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.plot(curve_df["confidence_score"], curve_df["calibrated_ape"] * 100, color="#4C72B0", linewidth=2)
for name, val in [("alta_min", thresholds["alta_min"]), ("moderada_min", thresholds["moderada_min"]),
                   ("baixa_min", thresholds["baixa_min"])]:
    ax.axvline(val, color="#C44E52", linestyle="--", linewidth=1)
    ax.text(val, ax.get_ylim()[1] * 0.92, name, rotation=90, fontsize=8, color="#C44E52",
            ha="right", va="top")
ax.set_xlabel("Confidence Score (0-100)")
ax.set_ylabel("APE calibrado (%, regressão isotônica sobre TEST)")
ax.set_title("Curva de calibração isotônica — score -> erro esperado (TEST)")
plt.tight_layout()
plt.savefig("reports/figures/15_isotonic_calibration_curve.png", dpi=110)
plt.show()
plt.close()

In [ ]:
import numpy as np

def matrix_heatmap(df, title, filename):
    grid = np.array([
        [df.loc[df["cell"] == "cobertura_alta__erro_baixo", "median_abs_error"].iloc[0],
         df.loc[df["cell"] == "cobertura_alta__erro_alto", "median_abs_error"].iloc[0]],
        [df.loc[df["cell"] == "cobertura_baixa__erro_baixo", "median_abs_error"].iloc[0],
         df.loc[df["cell"] == "cobertura_baixa__erro_alto", "median_abs_error"].iloc[0]],
    ])
    labels = np.array([
        [f"ALTA\n\${grid[0,0]:,.0f}\nn={df.loc[df['cell']=='cobertura_alta__erro_baixo','n'].iloc[0]}",
         f"MEDIA\n\${grid[0,1]:,.0f}\nn={df.loc[df['cell']=='cobertura_alta__erro_alto','n'].iloc[0]}"],
        [f"MEDIA\n\${grid[1,0]:,.0f}\nn={df.loc[df['cell']=='cobertura_baixa__erro_baixo','n'].iloc[0]}",
         f"BAIXA\n\${grid[1,1]:,.0f}\nn={df.loc[df['cell']=='cobertura_baixa__erro_alto','n'].iloc[0]}"],
    ])
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(grid, cmap="RdYlGn_r")
    ax.set_xticks([0, 1], ["Erro esperado baixo", "Erro esperado alto"])
    ax.set_yticks([0, 1], ["Cobertura alta", "Cobertura baixa"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, labels[i, j], ha="center", va="center", fontsize=10, color="black")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="MAE mediano ($)")
    plt.tight_layout()
    plt.savefig(filename, dpi=110)
    plt.show()
    plt.close()

matrix_heatmap(matrix_test_df, "Matriz 2x2 de confiança — TEST (calibração)",
               "reports/figures/15_confidence_matrix_test.png")
matrix_heatmap(matrix_val_df, "Matriz 2x2 de confiança — VAL (verificação)",
               "reports/figures/15_confidence_matrix_val.png")

## Resultado

**Cortes de categoria** (calibrados a partir da curva isotônica, não escolhidos "porque parecem
bons"): score mínimo 88 (Alta confiança), 55 (Confiança moderada), 17 (Baixa confiança), abaixo de 17
= Muito baixa confiança / revisão. Não ficaram em 80/60/40 (o exemplo ilustrativo do pedido original)
— ficaram onde a curva calibrada de erro realmente muda de regime nesta execução.

**4 categorias finas — calibração em TEST (n=2.644):**

| Categoria | Score mínimo | APE calibrado (teto) | n em TEST | MAE mediano em TEST |
|---|---|---|---|---|
| Alta confiança | 88 | 16,3% (P25) | 222 | \$62.483 |
| Confiança moderada | 55 | 20,9% (P50) | 960 | \$40.653 |
| Baixa confiança | 17 | 23,1% (P75) | 887 | \$78.478 |
| Muito baixa / revisão | < 17 | — | 575 | \$126.801 |

**Matriz 2×2 didática — TEST:** cobertura alta/erro esperado baixo (ALTA) MAE mediano \$43.038
(n=1.100); cobertura alta/erro alto (MÉDIA) \$61.869 (n=222); cobertura baixa/erro baixo (MÉDIA)
\$67.254 (n=417); cobertura baixa/erro alto (BAIXA) \$127.078 (n=905).

**Verificação em VAL (n=3.868) — a confiança prometida se sustenta?** Todas as categorias com teto
definido ficaram **dentro** do erro prometido em TEST (nenhuma promessa quebrada no sentido "prometeu
menos erro do que entregou"): Alta confiança observou 12,3% de APE mediano (teto 16,3%), Confiança
moderada 10,3% (teto 20,9%), Baixa confiança 10,8% (teto 23,1%). A **matriz 2×2 em VAL** confirma o
mesmo padrão de forma mais robusta: célula ALTA com o melhor MAE mediano de todas.

**Checagem de monotonicidade (o resultado honesto):** `is_monotonic_by_category` = **False**. A ORDEM
fina das 4 categorias não se sustentou perfeitamente em VAL — "Alta confiança" (12,3% de APE mediano
observado) teve erro **maior** que "Confiança moderada" (10,3%) e "Baixa confiança" (10,8%), mesmo com
o teto de erro de cada categoria individualmente respeitado. Isso é consistente com
`docs/09_confidence_matrix.md` (seção 6, "Limitação observada"): a categorização fina de 4 níveis é
sensível ao ruído de segmentos com poucas combinações distintas (~37 células level0), parte do efeito
vindo do viés já documentado de que a banda `Entry` tem MAPE historicamente mais alto que
`Standard`/`Premium` (superestimação em imóveis baratos), não um MAPE baixo só por o imóvel ser
barato — e parte das células "Alta confiança" desta calibração vem de bandas mais baratas.

![curva de calibração isotônica](../reports/figures/15_isotonic_calibration_curve.png)
![matriz 2x2 em test](../reports/figures/15_confidence_matrix_test.png)
![matriz 2x2 em val](../reports/figures/15_confidence_matrix_val.png)

**Decisão (mesma leitura de `docs/09_confidence_matrix.md`):** mantido como está, documentado
abertamente. Uso recomendado: confiar no **teto de erro por categoria** (sempre respeitado em VAL) e
na **matriz 2×2** (mais robusta) para leitura rápida; tratar a ordem fina de 4 categorias como
indicativa, não como um ranking perfeitamente monotônico — texto explícito nas respostas da API/portal.
Este notebook não toma nenhuma decisão de modelo/feature (P4) — apenas produz e verifica os artefatos
de confiança já congelados.

## Artefatos

`src/evaluation/confidence.py`, `scripts/build_confidence_matrix.py`,
`artifacts/confidence_calibration.json`, `reports/confidence_matrix_report.json`,
`reports/figures/15_isotonic_calibration_curve.png`,
`reports/figures/15_confidence_matrix_test.png`, `reports/figures/15_confidence_matrix_val.png`.
Metodologia completa: `docs/09_confidence_matrix.md`.